<a href="https://colab.research.google.com/github/BrenoBragaGalvao/mestrado-arvores-urbanas/blob/main/Segmentacao_area_completa_v2_experimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentação de arvores e análise

## Preparação dos dados para segmentação:

Instalação das bibliotecas neessárias:

In [ ]:
!pip install rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Importação das funções e pacotes

In [ ]:
from skimage.segmentation import felzenszwalb, slic, quickshift, watershed
from skimage.segmentation import mark_boundaries
from skimage.util import img_as_float
from skimage.filters import sobel
from skimage.color import rgb2gray
from skimage import color
import rasterio
from matplotlib import pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
from rasterio.plot import show

Abrir a imagem com o rasterio, convertendo num numpy array:

In [ ]:
path_img = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_RGB_NEW.tif'
src_rgb = rasterio.open(path_img)
im = src_rgb.read()
im  = im.transpose([1,2,0])
'''R = src_rgb.read(3)
G = src_rgb.read(2)
B = src_rgb.read(1)


R = R / 10000
G = G / 10000
B = B / 10000

R = R * 255
G = G * 255
B = B * 255

R = R.astype('uint8')
G = G.astype('uint8')
B = B.astype('uint8')

im = np.dstack((R,G,B))'''

Plotando a imagem com o matplotlib:

In [ ]:
plt.figure(figsize=[12,12])
plt.imshow(im)
plt.axis('off')

Abrir o poligono da área de interesse:

In [ ]:
AOI = gpd.read_file('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_new.shp')

## Segmentação:

Aplicação do quickshift (Ver artigos relacionados)

In [ ]:
segments_quick2 = quickshift(im, kernel_size=3, max_dist=5, ratio=0.04)

In [ ]:
plt.figure(figsize=[16,16])
plt.imshow(mark_boundaries(im, segments_quick2))
plt.axis('off')

In [ ]:
segments_quick2 = segments_quick2.astype('uint16')

Conversão para poligono e extração das caracteristicas espectrais:

In [ ]:
from rasterio import features
from shapely.geometry import shape
import geopandas as gpd
import shapely
import json
from rasterio.mask import mask
import pandas as pd

Primeiro, converter os segmentos em poligonos usando geopandas e rasterio:

In [ ]:
find_shapes = features.shapes(segments_quick2, transform=src_rgb.transform)

In [ ]:
polygons = [shapely.geometry.Polygon(shape[0]["coordinates"][0]) for shape in find_shapes]

In [ ]:
crs = src_rgb.crs
Poly_gdf = gpd.GeoDataFrame(crs=crs, geometry=polygons)

Salvar os poligonos em um JSON:

In [ ]:
Poly_gdf.to_file("polygons.json")

In [ ]:
Poly_gdf.boundary.plot()

Obter somente os poligonos da área de interesse:

In [ ]:
clip_poly = Poly_gdf.clip(AOI)

In [ ]:
clip_poly.boundary.plot()

Extrair a média espectral de cada segmento:

In [ ]:
clip_poly = clip_poly.reset_index(drop=True)

In [ ]:
json_dataset = json.loads(clip_poly.to_json())['features']

In [ ]:
path_img_mult = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/AOI_NEW_MULTI.tif'
src = rasterio.open(path_img_mult)

X = []
Y = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=0)

  out_image = out_image.transpose([1,2,0])

  R_mean = out_image[:,:,0][np.nonzero(out_image[:,:,0])].mean()
  G_mean = out_image[:,:,1][np.nonzero(out_image[:,:,1])].mean()
  B_mean = out_image[:,:,2][np.nonzero(out_image[:,:,2])].mean()
  N_mean = out_image[:,:,3][np.nonzero(out_image[:,:,3])].mean()
  R_std = out_image[:,:,0][np.nonzero(out_image[:,:,0])].std()
  G_std = out_image[:,:,1][np.nonzero(out_image[:,:,1])].std()
  B_std = out_image[:,:,2][np.nonzero(out_image[:,:,2])].std()
  N_std = out_image[:,:,3][np.nonzero(out_image[:,:,3])].std()




  X.append([R_mean,G_mean,B_mean,N_mean,R_std,G_std,B_std,N_std])

In [ ]:
X = np.array(X)

## Aplicação do K-Means

Geramos o dataframe com a informação das médias espectrais por segmentos:

In [ ]:
df = pd.DataFrame(X, columns=['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std'])

In [ ]:
df

In [ ]:
df.dropna(inplace=True)

Aplicamos o Kmeans:

In [ ]:
from sklearn.cluster import KMeans

# Apply K-Means clustering
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10) # You can change the number of clusters
kmeans_clusters = kmeans.fit_predict(df[['R_Mean','G_Mean','B_Mean','N_Mean','R_Std', 'G_Std', 'B_Std', 'N_Std']])

# Add K-Means cluster labels to the DataFrame
df['kmeans_cluster'] = kmeans_clusters

display(df.head())

Aplicamos o resultado do Kmeans para os segmentos e visualizamos:

In [ ]:
clip_poly['kmeans_cluster'] = df['kmeans_cluster']

In [ ]:
import matplotlib.pyplot as plt

# Plot the GeoDataFrame, colored by cluster
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
clip_poly.plot(column='kmeans_cluster', ax=ax, legend=True, cmap='magma') # Changed colormap to 'viridis'
plt.title('Polygons Colored by kmeans_cluster')
plt.axis('off')
plt.show()

Salvamos o resultado em um JSON:

In [ ]:
Poly_gdf.to_file('polygons.json')

In [ ]:
clip_poly[clip_poly['kmeans_cluster'] == 3].to_file('polygons_arvores.json')

Selecionamos somente o cluster de arvores:

In [ ]:
import rasterio.plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
clip_poly[clip_poly['kmeans_cluster'] == 3].boundary.plot(ax=ax, color='red')
plt.title('Original Image')
plt.axis('off')
plt.show()

In [ ]:
path_grama = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Grama.gpkg'
path_arvore = '/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/Arvores.gpkg'

In [ ]:
gdf_grama = gpd.read_file(path_grama)
gdf_arvore = gpd.read_file(path_arvore)

In [ ]:
gdf_grama['Classe'] = 'grama'
gdf_arvore['Classe'] = 'arvore'

In [ ]:
gdf_full = pd.concat([gdf_grama, gdf_arvore])
gdf_full.to_crs(clip_poly.crs, inplace=True)

In [ ]:
gdf_arvore_seg = clip_poly[clip_poly['kmeans_cluster'] == 3]

In [ ]:
gdf_arvore_seg_filter = gdf_arvore_seg.sjoin(gdf_full, how='inner', predicate='intersects')

In [ ]:
gdf_arvore_seg_filter.plot()

In [ ]:
gdf_arvore_seg_filter

In [ ]:
src = rasterio.open(path_img_mult)

In [ ]:
img = src.read()

In [ ]:
img.shape

In [ ]:
nir = img[3]
red = img[0]

In [ ]:
nir.shape

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(nir, cmap='gray')
plt.title('NIR Band')
plt.axis('off')
plt.show()

In [ ]:
import numpy as np

# Calculate NDVI
# Avoid division by zero by adding a small epsilon to the denominator
ndvi = (nir.astype(float) - red.astype(float)) / (nir.astype(float) + red.astype(float) + 1e-8)

# Plot NDVI
plt.figure(figsize=(10, 10))
plt.imshow(ndvi, cmap='RdYlGn', vmin=0, vmax=1) # RdYlGn is a common colormap for NDVI
plt.colorbar(label='NDVI Value')
plt.title('Normalized Difference Vegetation Index (NDVI)')
plt.axis('off')
plt.show()

In [ ]:
nir.max()

In [ ]:
from skimage.feature import canny
from skimage.util import img_as_ubyte

# Ensure nir is in the correct format (uint8) for Canny
nir_uint = nir/10000

# Apply Canny edge detection
edges = canny(nir_uint, sigma=1)

# Plot the edges
plt.figure(figsize=(10, 10))
plt.imshow(edges, cmap='gray')
plt.title('Canny Edges of NIR Band')
plt.axis('off')
plt.show()

In [ ]:
img_rgb = img[0:3]

In [ ]:
img_rgb = img_rgb.transpose([1,2,0])

In [ ]:
from skimage.color import rgb2hsv

# Convert RGB to HSV
img_hsv = rgb2hsv(img_rgb)

# Extract the intensity (Value) channel, which is the third channel (index 2)
intensity = img_hsv[:, :, 2]

# Display the intensity image
plt.figure(figsize=(10, 10))
plt.imshow(intensity, cmap='gray')
plt.title('Intensity Channel (HSV)')
plt.axis('off')
plt.show()

In [ ]:
# Convert edges (boolean) to float for stacking
edges_float = edges.astype(np.float32)

# Stack intensity and edges as two separate bands
# The order here is intensity first, then edges
stacked_bands = np.stack([intensity, edges_float, ndvi], axis=0)

# Get metadata from the original source (src) for writing the new TIFF
profile = src.profile

# Update profile for the new stacked image
# Change the number of bands to 2 and data type to float32
profile.update(
    dtype=rasterio.float32,
    count=3,
    nodata=None
)

# Define the output path for the new TIFF file
output_filepath = 'features_stack.tif'

# Write the stacked image to a new TIFF file
with rasterio.open(output_filepath, 'w', **profile) as dst:
    dst.write(stacked_bands)


In [ ]:
src = rasterio.open(output_filepath)
json_dataset = json.loads(gdf_arvore_seg_filter.to_json())['features']

X = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=-999)
  classe_values = json_dataset[i]['properties']['Classe']

  mean_intensity = out_image[0][out_image[0] != -999].mean()
  mean_edges = out_image[1][out_image[1] != -999].mean()
  mean_ndvi = out_image[2][out_image[2] != -999].mean()
  X.append([mean_intensity, mean_edges, mean_ndvi,classe_values])


In [ ]:
df = pd.DataFrame(X, columns=['mean_intensity','mean_edges','mean_ndvi','classe'])

In [ ]:
import seaborn as sns

# Create a pairplot with 'classe' as hue
sns.pairplot(df, hue='classe')
plt.show()

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Separate features (X) and target (y)
X = df.drop('classe', axis=1) # Features are all columns except 'classe'
y = df['classe'] # Target is the 'classe' column

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Random Forest Classifier
# You can adjust parameters like n_estimators (number of trees) as needed
rf_classifier = RandomForestClassifier(n_estimators=500, random_state=42)

# Train the model
rf_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test)

# Evaluate the model
print("Random Forest Classifier Report:")
print(classification_report(y_test, y_pred))
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}")

In [ ]:
# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=rf_classifier.classes_)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_classifier.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

In [ ]:
src = rasterio.open(output_filepath)
json_dataset = json.loads(gdf_arvore_seg.to_json())['features']

X_pred = []
for i in range(len(json_dataset)):
  coords = json_dataset[i]['geometry']
  out_image, out_transform = mask(src, [coords], crop=True, nodata=-999)

  mean_intensity = out_image[0][out_image[0] != -999].mean()
  mean_edges = out_image[1][out_image[1] != -999].mean()
  mean_ndvi = out_image[2][out_image[2] != -999].mean()
  X_pred.append([mean_intensity, mean_edges, mean_ndvi])

In [ ]:
df_pred = pd.DataFrame(X_pred, columns=['mean_intensity','mean_edges','mean_ndvi'])

In [ ]:
gdf_arvore_seg['predicted_classe'] = rf_classifier.predict(df_pred)

In [ ]:
gdf_arvore_seg

In [ ]:
gdf_arvore_seg['area'] = gdf_arvore_seg.area

In [ ]:
tree_polys = gdf_arvore_seg.copy()

## Seleção das arvores e aplicação da Alometria:

Selecionamos os segmentos que pertence ao cluster das arvores:

In [ ]:
tree_polys

Aplicamos a formula da alometria:

In [ ]:
import numpy as np

#C1
a1 = 0.5
b1 = 0.7
tree_polys['Power_tree_size_C1'] = a1 * (tree_polys['area'] ** b1)
tree_polys['Linear_tree_size_C1'] = a1 + (tree_polys['area'] * b1)


#C2
a2 = 0.6
b2 = 0.8
tree_polys['Power_tree_size_C2'] = a2 * (tree_polys['area'] ** b2)
tree_polys['Linear_tree_size_C2'] = a2 + (tree_polys['area'] * b2)


#C3
a3 = 2.5
b3 = 0.4
tree_polys['Power_tree_size_C3'] = a3 * (tree_polys['area'] ** b3)
tree_polys['Linear_tree_size_C3'] = a3 + (tree_polys['area'] * b3)

#C4
a4 = 4
b4 = 0.5
tree_polys['Power_tree_size_C4'] = a4 * (tree_polys['area'] ** b4)
tree_polys['Linear_tree_size_C4'] = a4 + (tree_polys['area'] * b4)

#C5

a5 = 0.4
b5 = 0.9
tree_polys['Power_tree_size_C5'] = a5 * (tree_polys['area'] ** b5)
tree_polys['Linear_tree_size_C5'] = a5 + (tree_polys['area'] * b5)



a6 = 1.35
b6 = 0.72
tree_polys['Power_tree_size_C6'] = a6 * (tree_polys['area'] ** b6)
tree_polys['Linear_tree_size_C6'] = a6 + (tree_polys['area'] * b6)

In [ ]:
columns = ['Power_tree_size_C1', 'Linear_tree_size_C1', 'Power_tree_size_C2', 'Linear_tree_size_C2',
           'Power_tree_size_C3', 'Linear_tree_size_C3', 'Power_tree_size_C4', 'Linear_tree_size_C4',
           'Power_tree_size_C5', 'Linear_tree_size_C5', 'Power_tree_size_C6', 'Linear_tree_size_C6']

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(data=tree_polys[columns])
plt.title('Box Plot of Tree Size Metrics for Different Constants')
plt.ylabel('Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(data=tree_polys['Power_tree_size_C3'])
plt.title('Box Plot of Tree Size Metrics for C3')
plt.ylabel('Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(tree_polys['Power_tree_size_C3'], bins=30, edgecolor='black')
plt.title('Histogram of Power_tree_size_C3')
plt.xlabel('Power_tree_size_C3')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

Obtemos somente arvores maiores que 7 metros:

In [ ]:
filter_tree = tree_polys[tree_polys['Power_tree_size_C3'] > 7]

Salvamos em um arquivo JSON:

In [ ]:
filter_tree.to_file('arvores_segmentadas.json')

## Análise das arvores proximas a linhas e postes:

Importação e visualização das linhas e postes:

In [ ]:
Postes = gpd.read_file('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1/postes.gpkg')

Arvores maiores que 7m a 1m de distancia ou menos de postes:

In [ ]:
Postes = Postes.to_crs(filter_tree.crs)

In [ ]:
Postes = Postes.clip(AOI)

### Arvores e postes a 1m

In [ ]:
postes_prox_tree_1m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=1)

In [ ]:
arvores_prox_postes_1m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_1m.boundary.plot(ax=ax, color='green')
postes_prox_tree_1m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

### Arvores e postes a 3m

In [ ]:
postes_prox_tree_3m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=3)

In [ ]:
arvores_prox_postes_3m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=3)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_3m.boundary.plot(ax=ax, color='green')
postes_prox_tree_3m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

### Arvores e postes a 5m

In [ ]:
postes_prox_tree_5m = gpd.sjoin_nearest(Postes, filter_tree, max_distance=5)

In [ ]:
arvores_prox_postes_5m = gpd.sjoin_nearest(filter_tree, Postes, max_distance=5)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
rasterio.plot.show(src_rgb, ax=ax)
arvores_prox_postes_5m.boundary.plot(ax=ax, color='green')
postes_prox_tree_5m.plot(ax=ax, color='blue')
plt.title('Original Image')
plt.axis('off')
plt.show()

## V2 — Auditoria de amostras e avaliação sem duplicação de segmentos

**Etapa 2.** As células anteriores preservam integralmente o código original; apenas as saídas antigas foram limpas nesta cópia experimental (continuam no notebook original). Execute as células anteriores para reconstruir `df` e `gdf_arvore_seg_filter`, depois execute as células a seguir.

A auditoria procura (1) segmentos repetidos pelo `sjoin(intersects)`, (2) rótulos conflitantes e (3) sobreposição de identificadores de segmentos entre treino e teste original. O novo teste é exploratório, usa somente `Area1`, exclui explicitamente os segmentos com rótulos conflitantes, mantém uma linha por segmento e executa cinco separações por grupo. Não permite inferir risco elétrico, nem resolve autocorrelação espacial entre segmentos vizinhos. Nenhum resultado novo foi executado ou validado ao publicar este notebook.


In [ ]:
# V2 — auditoria da independência das amostras (não modifica o RF original)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

atributos_v2 = ["mean_intensity", "mean_edges", "mean_ndvi"]
assert all(c in df.columns for c in atributos_v2 + ["classe"])
assert len(df) == len(gdf_arvore_seg_filter), "Linhas desalinhadas entre df e GeoDataFrame."
assert np.array_equal(df["classe"].astype(str).to_numpy(),
                      gdf_arvore_seg_filter["Classe"].astype(str).to_numpy()), (
    "Classes ou ordem das linhas não coincidem; interrompa a auditoria."
)

# O spatial join preserva o índice do segmento da tabela à esquerda.
segmentos_v2 = pd.Series(gdf_arvore_seg_filter.index.to_numpy(), name="segmento")
pares_v2 = pd.DataFrame({
    "segmento": segmentos_v2,
    "classe": df["classe"].astype(str).reset_index(drop=True)
})
conflitos_v2 = pares_v2.groupby("segmento")["classe"].nunique()
ids_conflitantes_v2 = conflitos_v2[conflitos_v2 > 1].index

indices_tr_v2, indices_te_v2 = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42
)
ids_em_ambos_v2 = set(segmentos_v2.iloc[indices_tr_v2]) & set(segmentos_v2.iloc[indices_te_v2])
print("=== Diagnóstico da divisão original ===")
print("Pares objeto-rótulo:", len(df))
print("Segmentos únicos:", pares_v2["segmento"].nunique())
print("Registros repetidos por segmento:", len(df) - pares_v2["segmento"].nunique())
print("Segmentos com classes conflitantes:", len(ids_conflitantes_v2))
print("Segmentos compartilhados no treino e teste originais:", len(ids_em_ambos_v2))
print("Registros de teste desses segmentos:", int(
    segmentos_v2.iloc[indices_te_v2].isin(ids_em_ambos_v2).sum()))
print("Nota: segmentos vizinhos podem continuar espacialmente correlacionados.")


In [ ]:
# V2 — teste exploratório: 1 linha por segmento, cinco divisões agrupadas
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

pares_unicos_v2 = pares_v2.loc[
    ~pares_v2["segmento"].isin(ids_conflitantes_v2)
].drop_duplicates("segmento")
print("Registros excluídos (repetições/conflitos):", len(df) - len(pares_unicos_v2))
X_agr_v2 = df.loc[pares_unicos_v2.index, atributos_v2].to_numpy(dtype=float)
y_agr_v2 = pares_unicos_v2["classe"].to_numpy()
g_agr_v2 = pares_unicos_v2["segmento"].to_numpy()
if len(pares_unicos_v2) < 10 or len(np.unique(y_agr_v2)) < 2:
    raise ValueError("Não há segmentos distintos suficientes com ambas as classes.")
if not np.isfinite(X_agr_v2).all():
    raise ValueError("Atributos não finitos; reveja as extrações antes da avaliação.")

linhas_v2 = []
cv_v2 = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
for rodada, (tr, te) in enumerate(cv_v2.split(X_agr_v2, y_agr_v2, groups=g_agr_v2), start=1):
    if len(np.unique(y_agr_v2[tr])) < 2 or len(np.unique(y_agr_v2[te])) < 2:
        print(f"Rodada {rodada} ignorada: falta classe no treino ou no teste.")
        continue
    modelo_auditoria_v2 = RandomForestClassifier(
        n_estimators=500, random_state=42, n_jobs=-1)
    modelo_auditoria_v2.fit(X_agr_v2[tr], y_agr_v2[tr])
    pred_v2 = modelo_auditoria_v2.predict(X_agr_v2[te])
    linhas_v2.append({
        "rodada": rodada,
        "n_treino": len(tr),
        "n_teste": len(te),
        "acuracia": accuracy_score(y_agr_v2[te], pred_v2),
        "acuracia_balanceada": balanced_accuracy_score(y_agr_v2[te], pred_v2),
        "f1_macro": f1_score(y_agr_v2[te], pred_v2, average="macro", zero_division=0)
    })
resultado_auditoria_v2 = pd.DataFrame(linhas_v2)
if resultado_auditoria_v2.empty:
    print("Não houve nenhuma rodada válida; métricas não disponíveis.")
else:
    print(resultado_auditoria_v2.to_string(index=False))
    print("\nMédia e desvio-padrão (rodadas válidas):")
    print(resultado_auditoria_v2[["acuracia", "acuracia_balanceada", "f1_macro"]]
          .agg(["mean", "std"]).to_string())
    print("\nATENÇÃO: não comparar diretamente com 0,87, pois os testes diferem.")


## E2 — validação espacial por blocos com a mesma Area1

**Etapa seguinte à auditoria E1.** Treino e teste agora são separados em *blocos geográficos*, não apenas por registros. Uma faixa de exclusão de **25 metros** ao redor dos polígonos de teste evita que amostras de treinamento fiquem imediatamente encostadas nos objetos avaliados.

Usa somente `df` e `gdf_arvore_seg_filter` produzidos acima (imagens/rótulos existentes na `Area1`). Remove segmentos repetidos e aqueles com rótulos conflitantes, como no E1, preservando a classificação original e seus resultados históricos. O RF mantém **500 árvores**, `random_state=42` e os três atributos originais.

Testa blocos de **250 m** e **500 m**, com até 10 partições por escala. Cada partição contém blocos inteiros no teste; se não houver duas classes ou dados suficientes, a partição é descartada e isso é relatado. A proporção final de objetos em teste pode diferir de 20%, pois o sorteio é por blocos.

**Atenção:** separações dentro da mesma área de interesse são validação espacial interna, não teste de generalização para outra cidade. As métricas só passam a existir após executar as células com os dados reais; não são substitutas das métricas históricas de 0,87.


In [ ]:
# E2a — preparar dados com objetos únicos e geometria espacial real (Area1)
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score
)

campos_e2 = ["mean_intensity", "mean_edges", "mean_ndvi"]
assert len(df) == len(gdf_arvore_seg_filter), "As linhas de df e do sjoin não coincidem."
assert all(x in df.columns for x in campos_e2 + ["classe"])
assert "Classe" in gdf_arvore_seg_filter.columns
assert gdf_arvore_seg_filter.crs is not None, "É necessário um CRS conhecido."
rotulos_df_e2 = df["classe"].astype(str).str.strip().str.lower().to_numpy()
rotulos_geo_e2 = gdf_arvore_seg_filter["Classe"].astype(str).str.strip().str.lower().to_numpy()
assert np.array_equal(rotulos_df_e2, rotulos_geo_e2), (
    "Ordem ou classes diferem entre df e gdf; auditoria interrompida."
)

dados_e2 = df[campos_e2].reset_index(drop=True).copy()
dados_e2["classe"] = rotulos_df_e2
# O índice à esquerda do sjoin é o identificador do segmento; pode se repetir.
dados_e2["segmento_id"] = gdf_arvore_seg_filter.index.astype(str).to_numpy()
dados_e2["geometry"] = list(gdf_arvore_seg_filter.geometry.to_numpy())
base_e2 = gpd.GeoDataFrame(
    dados_e2, geometry="geometry", crs=gdf_arvore_seg_filter.crs
)
n_classes_e2 = base_e2.groupby("segmento_id")["classe"].nunique()
ids_conflitantes_e2 = set(n_classes_e2[n_classes_e2 > 1].index)
qtd_pares_e2 = len(base_e2)
base_e2 = (base_e2.loc[~base_e2["segmento_id"].isin(ids_conflitantes_e2)]
           .drop_duplicates(subset=["segmento_id"]).copy())
for campo_e2 in campos_e2:
    base_e2[campo_e2] = pd.to_numeric(base_e2[campo_e2], errors="coerce")
base_e2[campos_e2] = base_e2[campos_e2].replace([np.inf, -np.inf], np.nan)
base_e2 = base_e2.dropna(subset=campos_e2 + ["geometry"]).reset_index(drop=True)
if base_e2.geometry.is_empty.any() or (~base_e2.geometry.is_valid).any():
    raise ValueError("Polígonos vazios ou inválidos; revisar geometrias antes da avaliação.")
# EPSG:31982 é o CRS métrico dos rasters originais da pesquisa.
base_e2 = base_e2.to_crs(epsg=31982)
if len(base_e2) < 20 or base_e2["classe"].nunique() != 2:
    raise ValueError("Dados insuficientes para avaliar as duas classes por blocos.")
print("=== E2 | Preparação para validação espacial ===")
print("Pares objeto-rótulo anteriores:", qtd_pares_e2)
print("Segmentos conflitantes excluídos:", len(ids_conflitantes_e2))
print("Objetos distintos e válidos:", len(base_e2))
print("Classes:", base_e2["classe"].value_counts().to_dict())
print("CRS métrico:", base_e2.crs)


In [ ]:
# E2b — separação espacial por blocos, embargo geométrico de 25 m e RF original
# Não baixa novos dados nem altera o modelo ou as variáveis do experimento histórico.
resultados_e2 = []
ignorados_e2 = []
tam_blocos_e2 = [250, 500]  # metros
faixa_exclusao_e2 = 25  # metros
centros_e2 = base_e2.geometry.centroid
xmin_e2, ymin_e2, _, _ = base_e2.total_bounds
X_e2 = base_e2[campos_e2].to_numpy(dtype=float)
y_e2 = base_e2["classe"].to_numpy()
classe_arvore_e2 = "arvore" if "arvore" in set(y_e2) else None

for tamanho_e2 in tam_blocos_e2:
    col_e2 = np.floor((centros_e2.x.to_numpy() - xmin_e2) / tamanho_e2).astype(int)
    lin_e2 = np.floor((centros_e2.y.to_numpy() - ymin_e2) / tamanho_e2).astype(int)
    grupos_e2 = np.array([f"{x}_{y}" for x, y in zip(col_e2, lin_e2)])
    total_blocos_e2 = len(np.unique(grupos_e2))
    if total_blocos_e2 < 5:
        ignorados_e2.append((tamanho_e2, 0, "menos de cinco blocos distintos"))
        continue

    splitter_e2 = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    for rodada_e2, (tr_e2, te_e2) in enumerate(
        splitter_e2.split(X_e2, y_e2, groups=grupos_e2), start=1
    ):
        assert not (set(grupos_e2[tr_e2]) & set(grupos_e2[te_e2]))
        assert not (
            set(base_e2.iloc[tr_e2]["segmento_id"])
            & set(base_e2.iloc[te_e2]["segmento_id"])
        )
        # Excluir do treino objetos a até 25m das geometrias do teste.
        faixa_e2 = base_e2.geometry.iloc[te_e2].buffer(faixa_exclusao_e2)
        area_teste_e2 = (faixa_e2.union_all() if hasattr(faixa_e2, "union_all")
                         else faixa_e2.unary_union)
        manter_e2 = ~base_e2.geometry.iloc[tr_e2].intersects(area_teste_e2).to_numpy()
        tr_seguro_e2 = tr_e2[manter_e2]
        if len(tr_seguro_e2) < 15 or len(te_e2) < 5:
            ignorados_e2.append((tamanho_e2, rodada_e2, "amostra insuficiente após exclusão"))
            continue
        if (len(np.unique(y_e2[tr_seguro_e2])) < 2
                or len(np.unique(y_e2[te_e2])) < 2):
            ignorados_e2.append((tamanho_e2, rodada_e2, "uma classe ausente em treino ou teste"))
            continue

        rf_e2 = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
        rf_e2.fit(X_e2[tr_seguro_e2], y_e2[tr_seguro_e2])
        previsto_e2 = rf_e2.predict(X_e2[te_e2])
        linha_e2 = {
            "bloco_m": tamanho_e2, "rodada": rodada_e2,
            "blocos_total": total_blocos_e2,
            "blocos_teste": len(np.unique(grupos_e2[te_e2])),
            "n_treino": len(tr_seguro_e2), "n_teste": len(te_e2),
            "n_excluidos_faixa": len(tr_e2) - len(tr_seguro_e2),
            "teste_arvore": int((y_e2[te_e2] == "arvore").sum()),
            "teste_grama": int((y_e2[te_e2] == "grama").sum()),
            "acuracia": accuracy_score(y_e2[te_e2], previsto_e2),
            "acuracia_balanceada": balanced_accuracy_score(y_e2[te_e2], previsto_e2),
            "f1_macro": f1_score(y_e2[te_e2], previsto_e2, average="macro", zero_division=0),
        }
        if classe_arvore_e2 is not None:
            linha_e2["precisao_arvore"] = precision_score(
                y_e2[te_e2], previsto_e2, pos_label="arvore", zero_division=0
            )
            linha_e2["recall_arvore"] = recall_score(
                y_e2[te_e2], previsto_e2, pos_label="arvore", zero_division=0
            )
            linha_e2["f1_arvore"] = f1_score(
                y_e2[te_e2], previsto_e2, pos_label="arvore", zero_division=0
            )
        resultados_e2.append(linha_e2)

tabela_e2 = pd.DataFrame(resultados_e2)
print("=== E2 | Resultados por separação geográfica ===")
if tabela_e2.empty:
    print("Nenhuma rodada foi viável com os blocos, as classes e a faixa definidos.")
else:
    print(tabela_e2.round(3).to_string(index=False))
    print("\nResumo descritivo por tamanho de bloco:")
    print(tabela_e2.groupby("bloco_m")[["acuracia", "acuracia_balanceada", "f1_macro"]]
          .agg(["count", "mean", "std"]).round(3).to_string())
print("\nRodadas ignoradas:", len(ignorados_e2))
for tamanho_e2, rodada_e2, motivo_e2 in ignorados_e2:
    print(f"  Bloco {tamanho_e2}m; rodada {rodada_e2}: {motivo_e2}")
print("\nINTERPRETAÇÃO: repetição de divisões na mesma área NÃO equivale a")
print("testes em novas cidades; desvios-padrão entre rodadas não são intervalos")
print("de confiança independentes nem comprovam ganho sobre a acurácia 0,87 histórica.")


## E3 — comparação pareada de classificadores (somente Area1)

**Objetivo:** comparar os resultados de **Random Forest (RF)**, **SVM de kernel RBF**, **Regressão Logística** e uma referência ingênua que sempre prevê a classe majoritária (**DummyClassifier**), utilizando **exatamente os mesmos registros de teste em cada rodada**.

- Reutiliza os segmentos únicos e as três características `mean_intensity` (brilho HSV), `mean_edges` (bordas Canny) e `mean_ndvi` (NDVI), preparados em **E2a**.
- Reproduz o protocolo do **E2b**: blocos de 250 e 500 m, 10 tentativas por escala, semente 42, 20% dos **blocos** reservados para teste e exclusão de objetos de treino a até 25 m da geometria de teste.
- SVM e regressão logística aplicam `StandardScaler` **ajustado exclusivamente no treino**; o RF usa os mesmos 500 estimadores do experimento original.
- Não há busca de hiperparâmetros no conjunto de teste. Os quatro métodos utilizam as mesmas entradas e divisão; são classificadores de **árvore versus gramínea**, não detectores completos comparáveis a YOLO sem anotações adicionais.
- Repetições de teste compartilham amostras; as diferenças pareadas são descritivas, **não** testes independentes de significância.
- A análise continua restrita à **mesma área Area1**; os dados atuais não permitem alegar generalização externa, validar altura real ou interferência nos cabos.

**Execução:** rode as células originais e **E2a**, depois as duas células E3 a seguir. A execução de **E2b** não é obrigatória, mas, se tiver sido concluída, o E3 também confere a consistência de suas métricas do RF. Nenhuma métrica nova é presumida antes de executar o Colab.


In [ ]:
# E3a — RF vs SVM vs regressão logística vs baseline, nas MESMAS partições espaciais de E2
# Pré-requisito: E2a concluída; não altera rf_classifier, df ou os dados originais.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
)
import numpy as np
import pandas as pd

if "base_e2" not in globals() or "campos_e2" not in globals():
    raise RuntimeError("Execute E2a para criar base_e2/campos_e2 antes de E3.")
assert base_e2.crs is not None and base_e2.crs.is_projected, (
    "A geometria de E3 deve usar um CRS projetado com unidades em metros."
)
assert len(base_e2) == base_e2["segmento_id"].nunique(), (
    "Há segmentos repetidos; verifique a preparação E2a."
)
assert set(base_e2["classe"].unique()) == {"arvore", "grama"}, (
    "Esperadas as classes 'arvore' e 'grama' em E3."
)

X_e3 = base_e2[campos_e2].to_numpy(dtype=float)
y_e3 = base_e2["classe"].to_numpy()
if not np.isfinite(X_e3).all():
    raise ValueError("Atributos não finitos em E3.")
centros_e3 = base_e2.geometry.centroid
xmin_e3, ymin_e3, _, _ = base_e2.total_bounds
resultados_e3, ignorados_e3 = [], []
protocolos_validos_e3 = []

for tamanho_e3 in [250, 500]:
    col_e3 = np.floor((centros_e3.x.to_numpy() - xmin_e3) / tamanho_e3).astype(int)
    lin_e3 = np.floor((centros_e3.y.to_numpy() - ymin_e3) / tamanho_e3).astype(int)
    grupos_e3 = np.array([f"{c}_{l}" for c, l in zip(col_e3, lin_e3)])
    n_blocos_e3 = len(np.unique(grupos_e3))
    if n_blocos_e3 < 5:
        ignorados_e3.append((tamanho_e3, 0, "menos de cinco blocos"))
        continue

    separador_e3 = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
    for rodada_e3, (treino_e3, teste_e3) in enumerate(
        separador_e3.split(X_e3, y_e3, groups=grupos_e3), start=1
    ):
        if set(grupos_e3[treino_e3]) & set(grupos_e3[teste_e3]):
            raise AssertionError("Vazamento de blocos geográficos entre treino e teste.")
        if (set(base_e2.iloc[treino_e3]["segmento_id"]) &
                set(base_e2.iloc[teste_e3]["segmento_id"])):
            raise AssertionError("Mesmo segmento em treino e teste.")
        # Mesmo embargo espacial aplicado em E2b.
        buffers_e3 = base_e2.geometry.iloc[teste_e3].buffer(25)
        zona_e3 = (buffers_e3.union_all() if hasattr(buffers_e3, "union_all")
                   else buffers_e3.unary_union)
        afastados_e3 = ~base_e2.geometry.iloc[treino_e3].intersects(zona_e3).to_numpy()
        treino_final_e3 = treino_e3[afastados_e3]

        if len(treino_final_e3) < 15 or len(teste_e3) < 5:
            ignorados_e3.append((tamanho_e3, rodada_e3, "amostra insuficiente"))
            continue
        if (len(np.unique(y_e3[treino_final_e3])) != 2
                or len(np.unique(y_e3[teste_e3])) != 2):
            ignorados_e3.append((tamanho_e3, rodada_e3, "faltou uma classe"))
            continue

        modelos_e3 = {
            "Dummy (maioria)": DummyClassifier(strategy="most_frequent"),
            "Regressão logística": make_pipeline(
                StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)
            ),
            "SVM (RBF)": make_pipeline(
                StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")
            ),
            "Random Forest": RandomForestClassifier(
                n_estimators=500, random_state=42, n_jobs=-1
            ),
        }
        protocolos_validos_e3.append((tamanho_e3, rodada_e3))
        for nome_e3, modelo_e3 in modelos_e3.items():
            modelo_e3.fit(X_e3[treino_final_e3], y_e3[treino_final_e3])
            previsoes_e3 = modelo_e3.predict(X_e3[teste_e3])
            resultados_e3.append({
                "bloco_m": tamanho_e3, "rodada": rodada_e3, "modelo": nome_e3,
                "n_treino": len(treino_final_e3), "n_teste": len(teste_e3),
                "n_excluidos_faixa": len(treino_e3) - len(treino_final_e3),
                "teste_arvore": int(np.sum(y_e3[teste_e3] == "arvore")),
                "teste_grama": int(np.sum(y_e3[teste_e3] == "grama")),
                "acuracia": accuracy_score(y_e3[teste_e3], previsoes_e3),
                "acuracia_balanceada": balanced_accuracy_score(
                    y_e3[teste_e3], previsoes_e3
                ),
                "f1_macro": f1_score(
                    y_e3[teste_e3], previsoes_e3, average="macro", zero_division=0
                ),
                "precisao_arvore": precision_score(
                    y_e3[teste_e3], previsoes_e3, pos_label="arvore", zero_division=0
                ),
                "recall_arvore": recall_score(
                    y_e3[teste_e3], previsoes_e3, pos_label="arvore", zero_division=0
                ),
                "f1_arvore": f1_score(
                    y_e3[teste_e3], previsoes_e3, pos_label="arvore", zero_division=0
                ),
            })

tabela_e3 = pd.DataFrame(resultados_e3)
print("=== E3 | Protocolo pareado entre classificadores ===")
print("Partições espaciais válidas:", len(protocolos_validos_e3))
print("Partições ignoradas:", len(ignorados_e3))
for b_e3, r_e3, motivo_e3 in ignorados_e3:
    print(f"  {b_e3} m | rodada {r_e3}: {motivo_e3}")

if tabela_e3.empty:
    print("Nenhuma partição permitiu comparar os modelos; não há métricas E3.")
else:
    n_modelos_e3 = tabela_e3.groupby(["bloco_m", "rodada"])["modelo"].nunique()
    assert (n_modelos_e3 == 4).all(), "Algum modelo não participou da mesma partição."
    print("\nMétricas por partição e algoritmo:")
    print(tabela_e3.round(3).to_string(index=False))
    print("\nResumo por escala e algoritmo — média e desvio-padrão descritivos:")
    resumo_e3 = tabela_e3.groupby(["bloco_m", "modelo"])[
        ["acuracia", "acuracia_balanceada", "f1_macro", "f1_arvore"]
    ].agg(["count", "mean", "std"]).round(3)
    print(resumo_e3.to_string())

    # Se E2b foi executado, suas métricas do RF devem coincidir na mesma partição.
    if ("tabela_e2" in globals() and isinstance(tabela_e2, pd.DataFrame)
            and not tabela_e2.empty):
        rf_e3 = tabela_e3.loc[tabela_e3["modelo"] == "Random Forest"]
        verif_e3 = rf_e3.merge(
            tabela_e2[["bloco_m", "rodada", "n_treino", "n_teste", "f1_macro"]],
            on=["bloco_m", "rodada"], how="outer",
            suffixes=("_e3", "_e2"), indicator=True
        )
        if not (verif_e3["_merge"] == "both").all():
            raise AssertionError("E2 e E3 não usaram as mesmas partições válidas.")
        assert np.array_equal(verif_e3["n_treino_e3"], verif_e3["n_treino_e2"])
        assert np.array_equal(verif_e3["n_teste_e3"], verif_e3["n_teste_e2"])
        assert np.allclose(verif_e3["f1_macro_e3"], verif_e3["f1_macro_e2"])
        print("\nConferência E2 × E3: mesmas divisões e métricas RF confirmadas.")

print("\nIMPORTANTE: não comparar diretamente estes valores com a acurácia 0,87")
print("histórica, calculada sob protocolo diferente.")


In [ ]:
# E3b — diferenças PAREADAS (mesmo teste por rodada); tabela e gráfico exploratório
# Não interpreta rodadas repetidas como observações estatisticamente independentes.
import matplotlib.pyplot as plt

if tabela_e3.empty:
    print("Sem métricas E3 para resumir ou representar graficamente.")
else:
    f1_pares_e3 = tabela_e3.pivot(
        index=["bloco_m", "rodada"], columns="modelo", values="f1_macro"
    )
    bal_pares_e3 = tabela_e3.pivot(
        index=["bloco_m", "rodada"], columns="modelo",
        values="acuracia_balanceada"
    )
    for alternativa_e3 in ["SVM (RBF)", "Regressão logística", "Dummy (maioria)"]:
        diferenca_e3 = f1_pares_e3["Random Forest"] - f1_pares_e3[alternativa_e3]
        print(f"\nDiferença de F1-macro por rodada: RF − {alternativa_e3}")
        print(diferenca_e3.groupby(level="bloco_m").agg(["count", "mean", "std"])
              .round(3).to_string())
    print("\nAcurácia balanceada: RF e concorrentes (mesmos testes):")
    print(bal_pares_e3.groupby(level="bloco_m").mean().round(3).to_string())

    for bloco_graf_e3, recorte_e3 in tabela_e3.groupby("bloco_m"):
        ordem_e3 = ["Dummy (maioria)", "Regressão logística", "SVM (RBF)",
                     "Random Forest"]
        media_e3 = recorte_e3.groupby("modelo")["f1_macro"].mean().reindex(ordem_e3)
        plt.figure(figsize=(9, 4))
        plt.bar(media_e3.index, media_e3.to_numpy())
        plt.ylim(0, 1)
        plt.ylabel("F1-macro médio nas partições válidas")
        plt.title(f"E3 — classificação com blocos espaciais de {bloco_graf_e3} m")
        plt.xticks(rotation=18, ha="right")
        plt.tight_layout()
        plt.show()

    print("\nConclusões dependem dos valores reais calculados após executar no Colab.")
    print("A comparação mede classificação árvore/grama na Area1 e não risco elétrico.")
